In [1]:
import glob
from general import *
from generation import *
from storage import *
from system import *
from copy import deepcopy

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
load_mw = Load_Data(years=2033).load_MW
#load_mw['load_MW'] = 20

In [4]:
Kfiles = glob.glob('/projects/wg-ASGARD/weather_data/KAFB/*2013*')

In [5]:
KAFB_site = Site('KAFB',Weather_Data(Kfiles),-7)

In [6]:
ASGARD_PV = PV_System(name='ASGARD_PV',
                      site= KAFB_site,
                      capacity_MW_DC= 143,
                      PV_array_type= 'fixed',
                      PV_tilt=30,
                      PV_azimuth=180,
                      ratio_DC2AC=1.2,
                      off_grid_operation=True,
                      power_priority_load_MW_AC = None)

pv = ASGARD_PV.power_timeseries.power_MW_AC.values

In [7]:
csp_config = {'thermal_power_MW_t': 100,
          'power_rating_MW_e': 50,
          'dni_des':950,
          'tower_height_m': 117,
          'receiver_width_m':5,
          'receiver_height_m':5,
          'accept_ang_y':180,
          'accept_ang_x':180,
          'heliostat_width_m':5,
          'heliostat_height_m':5,
          'field_max_scaled_rad':8,
          'layout_method':'Radial Stagger',
          'field_shape':'Hexagon',
          'interaction_limit':50,
          'row_spacing_x':1.5,
          'row_spacing_y':1.5,
          'receiver_dT': 1,
          'media_cp': 1,
          'optical_height_m': 1,}

In [8]:
ASGARD_CSP = CSP_System(name='ASGARD_CSP',
                        site = KAFB_site,
                        config = csp_config,
                        off_grid_operation = True)
pah = ASGARD_CSP.pah

In [9]:
ASGARD_TES = TES_System(name='ASGARD_TES',
                        site = KAFB_site,
                        capacity_MWh_e = 570,
                        power_rating_MW_e = 50,
                        power_minimum_MW_e = 0,
                        percent_discharge_depth = 95,
                        percent_heat_loss_daily = 1,
                        charge_rate_CSP_MW_t = .6*143,
                        charge_efficiency_t2TES = .98,
                        charge_rate_resistive_MW_e = None,
                        charge_efficiency_e2TES = .98,
                        systems_charging = ['ASGARD_CSP', 'ASGARD_PV2','ASGARD_PV', 'NSTTF_PV', 'Foxtail_PV'],
                        start_full = True,
                        off_grid_operation = True,
                        DOE_2030_targets = False,
                        dT_s = 200, # [C] dT across storage bins
                        cp_s = 1.15, # [kJ/kgK] storage media specific heat
                        rho_s = 2000) # [kg/m3] media bulk density)

t2e = ASGARD_TES.t2e_values

In [10]:
power_csp = Power_System(
    name="power_csp",
    site = KAFB_site,
    power_type="thermal",
    energy_type="thermal",
    capacity_MW = 50,
    to_load = False,
    off_grid_operation= True,
    power_timeseries= pah,
    power_priority_load_MW = None,
    capex=150000000,  # USD
    opex=5000000,  # USD/year
    land_area=500  # Acres
)

In [11]:
storage_tes = Storage_System(name='storage_tes',
                        site = KAFB_site,
                        capacity_MWh = 570,
                        power_type = 'thermal',
                        energy_type = 'thermal',
                        power_rating_MW = 50,
                        power_minimum_MW = 0,
                        baseload = False, #change
                        percent_discharge_depth = 95,
                        percent_loss_daily = 1,
                        charge_rate_MW = .6*143,
                        charge_efficiency = [0.98],
                        conversion_values = t2e,
                        off_grid_operation = True,
                        start_full = True,
                        systems_charging = ['power_csp'])

In [12]:
power_pv = Power_System(
    name="power_pv",
    site = KAFB_site,
    power_type="electric",
    energy_type="electric",
    capacity_MW = 143,
    to_load = True, # change
    off_grid_operation= True,
    power_timeseries= pv,
    power_priority_load_MW = None,
    capex=150000000,  # USD
    opex=5000000,  # USD/year
    land_area=500  # Acres
)

In [13]:
storage_bes = Storage_System(name='storage_bes',
                        site = KAFB_site,
                        capacity_MWh = 30,
                        power_type = 'thermal',
                        energy_type = 'thermal',
                        power_rating_MW = 10,
                        power_minimum_MW = 0,
                        baseload = False,
                        percent_discharge_depth = 80,
                        percent_loss_daily = 0,
                        charge_rate_MW = 0,
                        charge_efficiency = [0.92],
                        conversion_values = [0.92] * 8760,
                        off_grid_operation = True,
                        start_full = True,
                        systems_charging = ['power_pv'])

In [14]:
ASGARD_systems = ['storage_tes', 'power_csp']
# ASGARD_systems = ['storage_bes', 'power_pv']
# ASGARD_systems = ['storage_tes', 'power_csp', 'storage_bes', 'power_pv']
# ASGARD_systems = ['ASGARD_PV','ASGARD_CSP','ASGARD_TES','ASGARD_BES']

In [15]:
systems=ASGARD_systems
ASGARD = System(load_MW = load_mw,
             systems_load_order = [globals()[sys] for sys in systems])

In [16]:
# ASGARD.system_metrics()

In [17]:
ts = ASGARD.timeseries

In [18]:
ts.columns

Index(['load_MW', 'target_load_MW', 'grid_to_load_MWh_e',
       'unmet_target_load_MWh_e', 'export_energy_MWh_e',
       'electricity_sale_in_hour', 'power_csp_power_MW',
       'power_csp_to_load_MWh', 'power_csp_to_grid_MWh',
       'power_csp_curtailed_MWh', 'power_csp_to_storage_tes_MWh',
       'storage_tes_MWh', 'storage_tes_to_load_MWh', 'storage_tes_loss_MWh',
       'KAFB_POI_MW', 'Power_System_to_storage_tes_MWh', 'unmet_load_MWh_e'],
      dtype='object')

In [19]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')
ts.head(50)

,load_MW,target_load_MW,grid_to_load_MWh_e,unmet_target_load_MWh_e,export_energy_MWh_e,electricity_sale_in_hour,power_csp_power_MW,power_csp_to_load_MWh,power_csp_to_grid_MWh,power_csp_curtailed_MWh,power_csp_to_storage_tes_MWh,storage_tes_MWh,storage_tes_to_load_MWh,storage_tes_loss_MWh,KAFB_POI_MW,Power_System_to_storage_tes_MWh,unmet_load_MWh_e
Date/time,,,,,,,,,,,,,,,,,
2033-01-01 00:30:00,61.873062,None,0.000000,0.000000,0,0.000000,0.000000,0,0,0,0,570.000000,0.000000,0.000000,0.000000,NaN,NaN
2033-01-01 01:30:00,62.962898,None,12.962898,NaN,0,3500.000000,0.000000,0,0,0,0,479.278288,50.000000,0.237500,50.000000,0.000000,12.962898
2033-01-01 02:30:00,62.851831,None,12.851831,NaN,0,3500.000000,0.000000,0,0,0,0,388.431581,50.000000,0.199699,50.000000,0.000000,12.851831
2033-01-01 03:30:00,62.807868,None,12.807868,NaN,0,3500.000000,0.000000,0,0,0,0,297.398650,50.000000,0.161846,50.000000,0.000000,12.807868
2033-01-01 04:30:00,64.001828,None,14.001828,NaN,0,3500.000000,0.000000,0,0,0,0,206.217739,50.000000,0.123916,50.000000,0.000000,14.001828
2033-01-01 05:30:00,64.496998,None,14.496998,NaN,0,3500.000000,0.000000,0,0,0,0,114.849726,50.000000,0.085924,50.000000,0.000000,14.496998
2033-01-01 06:30:00,63.874565,None,16.694375,NaN,0,3302.613327,0.000000,0,0,0,0,28.500000,47.180190,0.047854,47.180190,0.000000,16.694375
2033-01-01 07:30:00,64.300318,None,64.300318,NaN,0,0.000000,0.000000,0,0,0,0,28.488125,0.000000,0.011875,0.000000,0.000000,64.300318
2033-01-01 08:30:00,63.425673,None,63.425673,NaN,0,0.000000,0.000000,0,0,0,0,28.476255,0.000000,0.011870,0.000000,0.000000,63.425673


In [20]:
# ASGARD.metrics